<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Lab: Custom Training Loops in Keras**


Estimated time needed: **30** minutes


In this lab, you will learn to implement a basic custom training loop in Keras. 


## Objectives

By the end of this lab, you will: 

- Set up the environment 

- Define the neural network model 

- Define the Loss Function and Optimizer 

- Implement the custom training loop 

- Enhance the custom training loop by adding an accuracy metric to monitor model performance 

- Implement a custom callback to log additional metrics and information during training


----


## Step-by-Step Instructions:


### Exercise 1: Basic custom training loop: 

#### 1. Set Up the Environment:

- Import necessary libraries. 

- Load and preprocess the MNIST dataset. 


In [1]:
#!pip install tensorflow numpy

In [2]:
import os
import warnings
import tensorflow as tf 
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Flatten, Input
from tensorflow.keras.callbacks import Callback
import numpy as np

# Suppress all Python warnings
warnings.filterwarnings('ignore')

# Set TensorFlow log level to suppress warnings and info messages
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Step 1: Set Up the Environment
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data() 
x_train, x_test = x_train / 255.0, x_test / 255.0 
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)


#### 2. Define the model: 

Create a simple neural network model with a Flatten layer followed by two Dense layers. 


In [3]:
# Step 2: Define the Model

model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(128, activation='relu'),
    Dense(10)
])


#### 3. Define Loss Function and Optimizer: 

- Use Sparse Categorical Crossentropy for the loss function. 
- Use the Adam optimizer. 


In [4]:
# Step 3: Define Loss Function and Optimizer

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True) 
optimizer = tf.keras.optimizers.Adam()


#### 4. Implement the Custom Training Loop: 

- Iterate over the dataset for a specified number of epochs. 
- Compute the loss and apply gradients to update the model's weights. 


In [5]:
# Step 4: Implement the Custom Training Loop

epochs = 2
# train_dataset = train_dataset.repeat(epochs)
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)
for epoch in range(epochs):
    print(f'Start of epoch {epoch + 1}')

    for step, (x_batch_train, y_batch_train) in enumerate(train_dataset):
        with tf.GradientTape() as tape:
            logits = model(x_batch_train, training=True)  # Forward pass
            loss_value = loss_fn(y_batch_train, logits)  # Compute loss

        # Compute gradients and update weights
        grads = tape.gradient(loss_value, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))

        # Logging the loss every 200 steps
        if step % 200 == 0:
            print(f'Epoch {epoch + 1} Step {step}: Loss = {loss_value.numpy()}')


Start of epoch 1
Epoch 1 Step 0: Loss = 2.4468040466308594
Epoch 1 Step 200: Loss = 0.4048105478286743
Epoch 1 Step 400: Loss = 0.20185646414756775
Epoch 1 Step 600: Loss = 0.20110371708869934
Epoch 1 Step 800: Loss = 0.14994405210018158
Epoch 1 Step 1000: Loss = 0.4717784523963928
Epoch 1 Step 1200: Loss = 0.19728951156139374
Epoch 1 Step 1400: Loss = 0.26622623205184937
Epoch 1 Step 1600: Loss = 0.19050216674804688
Epoch 1 Step 1800: Loss = 0.1492675244808197
Start of epoch 2
Epoch 2 Step 0: Loss = 0.07188968360424042
Epoch 2 Step 200: Loss = 0.20985358953475952
Epoch 2 Step 400: Loss = 0.13254745304584503
Epoch 2 Step 600: Loss = 0.06774336844682693
Epoch 2 Step 800: Loss = 0.07294921576976776
Epoch 2 Step 1000: Loss = 0.31869107484817505
Epoch 2 Step 1200: Loss = 0.14479057490825653
Epoch 2 Step 1400: Loss = 0.14963579177856445
Epoch 2 Step 1600: Loss = 0.15956707298755646
Epoch 2 Step 1800: Loss = 0.07916229218244553


### Exercise 2: Adding Accuracy Metric:

Enhance the custom training loop by adding an accuracy metric to monitor model performance. 

#### 1. Set Up the Environment: 

Follow the setup from Exercise 1. 


In [6]:
import tensorflow as tf 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense, Flatten 

# Step 1: Set Up the Environment
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize the pixel values to be between 0 and 1
x_train, x_test = x_train / 255.0, x_test / 255.0 

# Create a batched dataset for training
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)


#### 2. Define the Model: 
Use the same model as in Exercise 1. 


In [7]:
# Step 2: Define the Model

model = Sequential([ 
    Flatten(input_shape=(28, 28)),  # Flatten the input to a 1D vector
    Dense(128, activation='relu'),  # First hidden layer with 128 neurons and ReLU activation
    Dense(10)  # Output layer with 10 neurons for the 10 classes (digits 0-9)
])


#### 3. Define the loss function, optimizer, and metric: 

- Use Sparse Categorical Crossentropy for the loss function and Adam optimizer. 

- Add Sparse Categorical Accuracy as a metric. 


In [8]:
# Step 3: Define Loss Function, Optimizer, and Metric

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)  # Loss function for multi-class classification
optimizer = tf.keras.optimizers.Adam()  # Adam optimizer for efficient training
accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy()  # Metric to track accuracy during training


#### 4. Implement the custom training loop with accuracy: 

Track the accuracy during training and print it at regular intervals. 


In [9]:
# Step 4: Implement the Custom Training Loop with Accuracy

epochs = 5  # Number of epochs for training

for epoch in range(epochs):
    print(f'Start of epoch {epoch + 1}')
    
    for step, (x_batch_train, y_batch_train) in enumerate(train_dataset):
        with tf.GradientTape() as tape:
            # Forward pass: Compute predictions
            logits = model(x_batch_train, training=True)
            # Compute loss
            loss_value = loss_fn(y_batch_train, logits)
        
        # Compute gradients
        grads = tape.gradient(loss_value, model.trainable_weights)
        # Apply gradients to update model weights
        optimizer.apply_gradients(zip(grads, model.trainable_weights))
        
        # Update the accuracy metric
        accuracy_metric.update_state(y_batch_train, logits)

        # Log the loss and accuracy every 200 steps
        if step % 200 == 0:
            print(f'Epoch {epoch + 1} Step {step}: Loss = {loss_value.numpy()} Accuracy = {accuracy_metric.result().numpy()}')
    
    # Reset the metric at the end of each epoch
    accuracy_metric.reset_state()


Start of epoch 1
Epoch 1 Step 0: Loss = 2.3812766075134277 Accuracy = 0.1875
Epoch 1 Step 200: Loss = 0.40193429589271545 Accuracy = 0.8356654047966003
Epoch 1 Step 400: Loss = 0.18514969944953918 Accuracy = 0.8699345588684082
Epoch 1 Step 600: Loss = 0.15192285180091858 Accuracy = 0.8856593370437622
Epoch 1 Step 800: Loss = 0.14823752641677856 Accuracy = 0.8979400992393494
Epoch 1 Step 1000: Loss = 0.42503243684768677 Accuracy = 0.9052510261535645
Epoch 1 Step 1200: Loss = 0.19775190949440002 Accuracy = 0.9119743704795837
Epoch 1 Step 1400: Loss = 0.24875414371490479 Accuracy = 0.9168004989624023
Epoch 1 Step 1600: Loss = 0.213565856218338 Accuracy = 0.9196596145629883
Epoch 1 Step 1800: Loss = 0.1403181254863739 Accuracy = 0.9237055778503418
Start of epoch 2
Epoch 2 Step 0: Loss = 0.0670999065041542 Accuracy = 1.0
Epoch 2 Step 200: Loss = 0.2167944610118866 Accuracy = 0.9625310897827148
Epoch 2 Step 400: Loss = 0.14944396913051605 Accuracy = 0.9592425227165222
Epoch 2 Step 600: Loss 

### Exercise 3: Custom Callback for Advanced Logging: 

Implement a custom callback to log additional metrics and information during training. 

#### 1. Set Up the Environment: 

Follow the setup from Exercise 1.


In [10]:
import tensorflow as tf 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense, Flatten 

# Step 1: Set Up the Environment
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize the pixel values to be between 0 and 1
x_train, x_test = x_train / 255.0, x_test / 255.0 

# Create a batched dataset for training
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)


#### 2. Define the Model: 

Use the same model as in Exercise 1. 


In [11]:
# Step 2: Define the Model

model = Sequential([
    Flatten(input_shape=(28, 28)),  # Flatten the input to a 1D vector
    Dense(128, activation='relu'),  # First hidden layer with 128 neurons and ReLU activation
    Dense(10)  # Output layer with 10 neurons for the 10 classes (digits 0-9)
])


#### 3. Define Loss Function, Optimizer, and Metric: 

- Use Sparse Categorical Crossentropy for the loss function and Adam optimizer. 

- Add Sparse Categorical Accuracy as a metric. 


In [12]:
# Step 3: Define Loss Function, Optimizer, and Metric

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)  # Loss function for multi-class classification
optimizer = tf.keras.optimizers.Adam()  # Adam optimizer for efficient training
accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy()  # Metric to track accuracy during training


#### 4. Implement the custom training loop with custom callback: 

Create a custom callback to log additional metrics at the end of each epoch.


In [13]:
from tensorflow.keras.callbacks import Callback

# Step 4: Implement the Custom Callback 
class CustomCallback(Callback):
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        print(f'End of epoch {epoch + 1}, loss: {logs.get("loss")}, accuracy: {logs.get("accuracy")}')


In [14]:
# Step 5: Implement the Custom Training Loop with Custom Callback

epochs = 2
custom_callback = CustomCallback()  # Initialize the custom callback

for epoch in range(epochs):
    print(f'Start of epoch {epoch + 1}')
    
    for step, (x_batch_train, y_batch_train) in enumerate(train_dataset):
        with tf.GradientTape() as tape:
            # Forward pass: Compute predictions
            logits = model(x_batch_train, training=True)
            # Compute loss
            loss_value = loss_fn(y_batch_train, logits)
        
        # Compute gradients
        grads = tape.gradient(loss_value, model.trainable_weights)
        # Apply gradients to update model weights
        optimizer.apply_gradients(zip(grads, model.trainable_weights))
        
        # Update the accuracy metric
        accuracy_metric.update_state(y_batch_train, logits)

        # Log the loss and accuracy every 200 steps
        if step % 200 == 0:
            print(f'Epoch {epoch + 1} Step {step}: Loss = {loss_value.numpy()} Accuracy = {accuracy_metric.result().numpy()}')
    
    # Call the custom callback at the end of each epoch
    custom_callback.on_epoch_end(epoch, logs={'loss': loss_value.numpy(), 'accuracy': accuracy_metric.result().numpy()})
    
    # Reset the metric at the end of each epoch
    accuracy_metric.reset_state()  # Use reset_state() instead of reset_states()


Start of epoch 1
Epoch 1 Step 0: Loss = 2.402977466583252 Accuracy = 0.125
Epoch 1 Step 200: Loss = 0.35235390067100525 Accuracy = 0.830690324306488
Epoch 1 Step 400: Loss = 0.18381014466285706 Accuracy = 0.8647911548614502
Epoch 1 Step 600: Loss = 0.16616196930408478 Accuracy = 0.8815515637397766
Epoch 1 Step 800: Loss = 0.1401849240064621 Accuracy = 0.8947799801826477
Epoch 1 Step 1000: Loss = 0.43280690908432007 Accuracy = 0.9023164510726929
Epoch 1 Step 1200: Loss = 0.1753363013267517 Accuracy = 0.9085657596588135
Epoch 1 Step 1400: Loss = 0.2597827911376953 Accuracy = 0.9134323596954346
Epoch 1 Step 1600: Loss = 0.23778966069221497 Accuracy = 0.9163023233413696
Epoch 1 Step 1800: Loss = 0.16560102999210358 Accuracy = 0.9206690788269043
End of epoch 1, loss: 0.026555288583040237, accuracy: 0.9227666854858398
Start of epoch 2
Epoch 2 Step 0: Loss = 0.07609550654888153 Accuracy = 1.0
Epoch 2 Step 200: Loss = 0.20972158014774323 Accuracy = 0.96159827709198
Epoch 2 Step 400: Loss = 0.0

### Exercise 4: Add Hidden Layers 

Next, you will add a couple of hidden layers to your model. Hidden layers help the model learn complex patterns in the data. 


In [15]:
from tensorflow.keras.layers import Input, Dense

# Define the input layer
input_layer = Input(shape=(28, 28))  # Input layer with shape (28, 28)

# Flatten the 2D images into 1D vectors before Dense layers
flatten = Flatten()(input_layer)

# Define hidden layers
hidden_layer1 = Dense(64, activation='relu')(flatten)  # First hidden layer with 64 neurons and ReLU activation
hidden_layer2 = Dense(64, activation='relu')(hidden_layer1)  # Second hidden layer with 64 neurons and ReLU activation


In the above code: 

`Flatten()` converts each 28×28 image into a 784-element 1D vector so it can be fed into Dense layers. 

`Dense(64, activation='relu')` creates a dense (fully connected) layer with 64 units and ReLU activation function. 

Each hidden layer takes the output of the previous layer as its input.


### Exercise 5: Define the output layer 

Finally, you will define the output layer. Suppose you are working on a binary classification problem, so the output layer will have one unit with a sigmoid activation function. 


In [16]:
output_layer = Dense(10, activation='softmax')(hidden_layer2)

In the above code: 

`Dense(10, activation='softmax')` creates the output layer with 10 neurons — one for each digit class (0–9). 

`softmax` activation ensures all 10 output values sum to 1, making them interpretable as class probabilities. 


### Exercise 6: Create the Model 

Now, you will create the model by specifying the input and output layers. 


In [17]:
model = Model(inputs=input_layer, outputs=output_layer)

In the above code: 

`Model(inputs=input_layer, outputs=output_layer)` creates a Keras model that connects the input layer to the output layer through the hidden layers. 


### Exercise 7: Compile the Model 

Before training the model, you need to compile it. You will specify the loss function, optimizer, and evaluation metrics. 


In [18]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # Correct for integer labels 0-9
    metrics=['accuracy']
)

In the above code: 

`optimizer='adam'` specifies the Adam optimizer, a popular choice for training neural networks. 

`loss='sparse_categorical_crossentropy'` is the correct loss for **multi-class classification** when labels are integers (0–9).

`metrics=['accuracy']` tells Keras to evaluate the model using accuracy during training. 


### Exercise 8: Train the Model 

You can now train the model on some training data. For this example, let's assume `X_train` is our training input data and `y_train` is the corresponding labels. 


In [19]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
import numpy as np

# Load and preprocess MNIST
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0  # Normalize pixel values to [0, 1]

# Train the model
history = model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=32
)

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 76s 26ms/step - accuracy: 0.9140 - loss: 0.2951
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 67s 18ms/step - accuracy: 0.9604 - loss: 0.1298
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 24s 9ms/step - accuracy: 0.9710 - loss: 0.0923
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - accuracy: 0.9777 - loss: 0.0714
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 26s 14ms/step - accuracy: 0.9810 - loss: 0.0588


In the above code: 

`X_train` and `y_train` are placeholders for your actual training data. 

`model.fit` trains the model for a specified number of epochs and batch size. 


### Exercise 9: Evaluate the Model 

After training, you can evaluate the model on test data to see how well it performs. 


In [20]:
# Example test data (in practice, use real dataset)
loss, accuracy = model.evaluate(x_test, y_test)

print(f'Test loss:     {loss:.4f}')
print(f'Test accuracy: {accuracy:.4f}')



313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9714 - loss: 0.0943
Test loss:     0.0943
Test accuracy: 0.9714


In the above code: 

`model.evaluate` computes the loss and accuracy of the model on test data. 

`X_test` and `y_test` are placeholders for your actual test data. 


## Practice Exercises 

### Exercise 1: Basic Custom Training Loop 

#### Objective: Implement a basic custom training loop to train a simple neural network on the MNIST dataset. 

#### Instructions: 

- Set up the environment and load the dataset. 

- Define the model with a Flatten layer and two Dense layers. 

- Define the loss function and optimizer. 

- Implement a custom training loop to iterate over the dataset, compute the loss, and update the model's weights. 


In [ ]:
# Write your code here
# Setting up the env
from tensorflow.keras.models import Sequential
(x_train, y_train),(x_test,y_test) = tf.keras.datasets.mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0
train_dataset = tf.data.Dataset.from_tensor_slices((x_train,y_train)).batch(32)

# Defining the model
model = Sequential([ 
    Flatten(input_shape=(28, 28)), 
    Dense(128, activation='relu'), 
    Dense(10) 
]) 

# Step 3: Defining loss function and optimizer
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam()

# Lastly custom training loop
for epoch in range(5): 
    for x_batch, y_batch in train_dataset: 
        with tf.GradientTape() as tape: 
            logits = model(x_batch, training=True) 
            loss = loss_fn(y_batch, logits) 
        grads = tape.gradient(loss, model.trainable_weights) 
        optimizer.apply_gradients(zip(grads, model.trainable_weights)) 
    print(f'Epoch {epoch + 1}: Loss = {loss.numpy()}')


Epoch 1: Loss = 0.055954765528440475
Epoch 2: Loss = 0.05673650652170181


### Exercise 2: Adding Accuracy Metric 

#### Objective: Enhance the custom training loop by adding an accuracy metric to monitor model performance. 

#### Instructions: 

1. Set up the environment and define the model, loss function, and optimizer. 

2. Add Sparse Categorical Accuracy as a metric. 

3. Implement the custom training loop with accuracy tracking.


In [ ]:
# Write your code here
# Step 1: Set Up the Environment
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data() 
x_train, x_test = x_train / 255.0, x_test / 255.0 
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32) 

# Step 2: Define the Model
model = Sequential([ 
    Flatten(input_shape=(28, 28)), 
    Dense(128, activation='relu'), 
    Dense(10) 
]) 

# Step 3: Define Loss Function, Optimizer and accuracy metric
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True) 
optimizer = tf.keras.optimizers.Adam()
accuracy_metric = tf.keras.metrics.SparseTopKCategoricalAccuracy() 

# Step 4: Implement the Custom Training Loop
for epoch in range(5): 
    for x_batch, y_batch in train_dataset: 
        with tf.GradientTape() as tape: 
            logits = model(x_batch, training=True) 
            loss = loss_fn(y_batch, logits) 
        grads = tape.gradient(loss, model.trainable_weights) 
        optimizer.apply_gradients(zip(grads, model.trainable_weights))
        accuracy_metric.update_state(y_batch, logits) 
    print(f'Epoch {epoch + 1}: Loss = {loss.numpy()}')
    accuracy_metric.update_state()

### Exercise 3: Custom Callback for Advanced Logging 

#### Objective: Implement a custom callback to log additional metrics and information during training. 

#### Instructions: 

1. Set up the environment and define the model, loss function, optimizer, and metric. 

2. Create a custom callback to log additional metrics at the end of each epoch. 

3. Implement the custom training loop with the custom callback. 


In [ ]:
# Write your code here
# Step 1: Set Up the Environment
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data() 
x_train, x_test = x_train / 255.0, x_test / 255.0 
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32) 

# Step 2: Define the Model
model = Sequential([ 
    Flatten(input_shape=(28, 28)), 
    Dense(128, activation='relu'), 
    Dense(10) 
]) 

# Step 3: Define Loss Function, Optimizer and accuracy metric
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True) 
optimizer = tf.keras.optimizers.Adam()
accuracy_metric = tf.keras.metrics.SparseTopKCategoricalAccuracy() 

# Step 4: Implement Custom Callback
class CustomCallback(Callback):
    def on_epoch_end(self,epoch,logs=None):
        print(f"End of epoch {epoch+1}, loss: {logs.get("loss")}, Accuracy: {logs.get("accuracy")}")

# Step 5: Implement the Custom Training Loop with Custom Callback
custom_callback = CustomCallback() 

for epoch in range(5): 
    for x_batch, y_batch in train_dataset: 
        with tf.GradientTape() as tape: 
            logits = model(x_batch, training=True) 
            loss = loss_fn(y_batch, logits) 
        grads = tape.gradient(loss, model.trainable_weights) 
        optimizer.apply_gradients(zip(grads, model.trainable_weights)) 
        accuracy_metric.update_state(y_batch, logits) 
    custom_callback.on_epoch_end(epoch, logs={'loss': loss.numpy(), 'accuracy': accuracy_metric.result().numpy()}) 
    accuracy_metric.reset_state()  # Updated method

### Exercise 5: Add Hidden layer and the Output Layer 

#### Objective: Add couple of hidden layer to help the model learn complex patterns in the data.
### Define the output layer of a neural network for a multi-class classification problem using the Keras Functional API. 

#### Instructions: 
- Define 2 hidden layers, Input layer(of shape 28,28) as the parameter for the first hidden layer)

- Using the `hidden_layer2` as the input, add a `Dense` output layer. 

- The output layer with `sigmoid function` as activation function


In [ ]:
# Write your code here
# Re-define layers for MNIST (28x28 images, 10 digit classes)
input_layer   = Input(shape=(28, 28))                          # Input: 28x28 grayscale images
flatten       = Flatten()(input_layer)                         # Flatten to 784-dim vector
hidden_layer1 = Dense(64, activation='relu')(flatten)          # First hidden layer
hidden_layer2 = Dense(64, activation='relu')(hidden_layer1)    # Second hidden layer
output_layer  = Dense(10, activation='softmax')(hidden_layer2) # Output: 10 classes (digits 0-9)

### Exercise 6: Create the Model 

#### Objective: Create a Keras Functional API model by connecting the input and output layers defined in the previous exercises. 

#### Instructions: 

- Use `tf.keras.Model` to create the model. 

- Pass `input_layer` as the `inputs` argument and `output_layer` as the `outputs` argument. 


In [ ]:
# Write your code here
# Create the model by specifying input and output layers
model = Model(inputs=input_layer, outputs=output_layer)

### Exercise 7: Compile the Model 

#### Objective: Configure the model for training by specifying the optimizer, loss function, and evaluation metric. 

#### Instructions: 

- Compile the model using the **Adam** optimizer. 

- Use **`sparse_categorical_crossentropy`** as the loss function
- Include **`accuracy`** as the evaluation metric. 


In [ ]:
# Write your code here
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # Correct loss for integer labels (0-9)
    metrics=['accuracy']
)

### Exercise 8: Train the Model 

#### Objective: Train the compiled model on the MNIST training dataset using `model.fit()`. 

#### Instructions: 

- Call `model.fit()` with the training data `x_train` and labels `y_train`. 

- Train for **5 epochs** with a **batch size of 32**. 


In [ ]:
# Write your code here
history = model.fit(
    x_train, y_train,        # Training data and labels
    epochs=5,                # Number of full passes over the training data
    batch_size=32,           # Number of samples per gradient update
)

print('Training complete.')

### Exercise 9: Evaluate the Model 

#### Objective: Assess the trained model's performance on unseen test data using `model.evaluate()`. 

#### Instructions: 

- Call `model.evaluate()` with the test data `x_test` and labels `y_test`. 

- Capture the returned **test loss** and **test accuracy**. 

- Print both values to summarise the model's generalisation performance. 


In [ ]:
# Write your code here
# Evaluate the trained model on the held-out test dataset
test_loss, test_accuracy = model.evaluate(x_test, y_test)

# Print the evaluation results
print(f'Test loss:     {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')

### Conclusion: 

Congratulations on completing this lab! You have now successfully created, trained, and evaluated a simple neural network model using the Keras Functional API. This foundational knowledge will allow you to build more complex models and explore advanced functionalities in Keras. 


Copyright © IBM Corporation. All rights reserved.
